# Healthcare Headline Sentiment Analysis

Devon Farber | Boston University MET CS688 coursework

Compares AFINN lexicon scores with DistilBERT sentiment classifications for healthcare AI headlines and exports an Excel comparison.

## Results and reproducibility
Recorded output: 20 headlines, with 5 exact label agreements and 15 disagreements. AFINN includes neutral while this DistilBERT model is binary, so disagreement is not an accuracy measure. Manual themes are tied to the original headline order and must be reassigned after fetching new headlines.

Saved outputs are retained from the submitted notebook. This portfolio packaging did not rerun the analysis.

See the repository README for setup and input requirements.


In [1]:
!pip install feedparser afinn transformers torch openpyxl -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 765.9 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 3.1 MB/s eta 0:00:00


In [2]:
import feedparser
import pandas as pd
from urllib.parse import quote

# Our two search topics
keywords = [
    "AI in healthcare",
    "AI medical diagnosis"
]

records = []

# Search Google News for each topic
for keyword in keywords:

    search_query = quote(keyword)

    url = (
        f"https://news.google.com/rss/search?"
        f"q={search_query}&hl=en-US&gl=US&ceid=US:en"
    )

    feed = feedparser.parse(url)

    # Take the first 10 results
    for entry in feed.entries[:10]:

        records.append({
            "Keyword": keyword,
            "Text": entry.title,
            "Link": entry.link
        })

# Turn everything into a table
df = pd.DataFrame(records)

df

,Keyword,Text,Link
0,AI in healthcare,"AI in Healthcare Poised to Transform Medicine,...",https://news.google.com/rss/articles/CBMirAFBV...
1,AI in healthcare,Experts Call for Stronger Healthcare AI Oversi...,https://news.google.com/rss/articles/CBMi1gFBV...
2,AI in healthcare,Americans want transparency when AI is used in...,https://news.google.com/rss/articles/CBMitgFBV...
3,AI in healthcare,Using AI in healthcare - wsaw.com,https://news.google.com/rss/articles/CBMiakFVX...
4,AI in healthcare,Wisconsin lawmakers weigh AI regulation in hea...,https://news.google.com/rss/articles/CBMikwFBV...
5,AI in healthcare,"AI Tracker: Forus lands $150M, reports $3B val...",https://news.google.com/rss/articles/CBMijwFBV...
6,AI in healthcare,The hidden costs of ungoverned AI: What leadin...,https://news.google.com/rss/articles/CBMiigFBV...
7,AI in healthcare,Ferris State’s Tech Week Grand Rapids sessions...,https://news.google.com/rss/articles/CBMipAFBV...
8,AI in healthcare,AI’s role in healthcare expanding as tech deve...,https://news.google.com/rss/articles/CBMiiwFBV...
9,AI in healthcare,AI | Will AI healthcare be China’s new global ...,https://news.google.com/rss/articles/CBMirgFBV...


In [3]:
from afinn import Afinn

# Create the AFINN sentiment analyzer
afinn = Afinn()

# Calculate a sentiment score for every headline
df["AFINN Score"] = df["Text"].apply(afinn.score)

# Turn the numerical score into an easy-to-read label
def afinn_label(score):
    if score > 0:
        return "Positive"
    elif score < 0:
        return "Negative"
    else:
        return "Neutral"

df["AFINN Sentiment"] = df["AFINN Score"].apply(afinn_label)

# Show the results
df[["Keyword", "Text", "AFINN Score", "AFINN Sentiment"]]

,Keyword,Text,AFINN Score,AFINN Sentiment
0,AI in healthcare,"AI in Healthcare Poised to Transform Medicine,...",-4.0,Negative
1,AI in healthcare,Experts Call for Stronger Healthcare AI Oversi...,1.0,Positive
2,AI in healthcare,Americans want transparency when AI is used in...,1.0,Positive
3,AI in healthcare,Using AI in healthcare - wsaw.com,0.0,Neutral
4,AI in healthcare,Wisconsin lawmakers weigh AI regulation in hea...,0.0,Neutral
5,AI in healthcare,"AI Tracker: Forus lands $150M, reports $3B val...",0.0,Neutral
6,AI in healthcare,The hidden costs of ungoverned AI: What leadin...,2.0,Positive
7,AI in healthcare,Ferris State’s Tech Week Grand Rapids sessions...,4.0,Positive
8,AI in healthcare,AI’s role in healthcare expanding as tech deve...,0.0,Neutral
9,AI in healthcare,AI | Will AI healthcare be China’s new global ...,2.0,Positive


In [4]:
from transformers import pipeline

# Load DistilBERT sentiment model
distilbert = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Run DistilBERT on every headline
distilbert_results = df["Text"].apply(lambda x: distilbert(x)[0])

# Save the label and confidence score
df["DistilBERT Sentiment"] = distilbert_results.apply(lambda x: x["label"])
df["DistilBERT Confidence"] = distilbert_results.apply(lambda x: round(x["score"], 4))

# Display the comparison
df[
    [
        "Text",
        "AFINN Score",
        "AFINN Sentiment",
        "DistilBERT Sentiment",
        "DistilBERT Confidence"
    ]
]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

,Text,AFINN Score,AFINN Sentiment,DistilBERT Sentiment,DistilBERT Confidence
0,"AI in Healthcare Poised to Transform Medicine,...",-4.0,Negative,NEGATIVE,0.9977
1,Experts Call for Stronger Healthcare AI Oversi...,1.0,Positive,POSITIVE,0.5517
2,Americans want transparency when AI is used in...,1.0,Positive,NEGATIVE,0.9589
3,Using AI in healthcare - wsaw.com,0.0,Neutral,NEGATIVE,0.9817
4,Wisconsin lawmakers weigh AI regulation in hea...,0.0,Neutral,NEGATIVE,0.7790
5,"AI Tracker: Forus lands $150M, reports $3B val...",0.0,Neutral,NEGATIVE,0.9967
6,The hidden costs of ungoverned AI: What leadin...,2.0,Positive,NEGATIVE,0.9987
7,Ferris State’s Tech Week Grand Rapids sessions...,4.0,Positive,POSITIVE,0.9912
8,AI’s role in healthcare expanding as tech deve...,0.0,Neutral,POSITIVE,0.7549
9,AI | Will AI healthcare be China’s new global ...,2.0,Positive,NEGATIVE,0.9566


In [5]:
# Show the full text of each headline
pd.set_option("display.max_colwidth", None)

for index, row in df.iterrows():
    print(f"{index}: {row['Text']}")
    print()

0: AI in Healthcare Poised to Transform Medicine, But Most Tools Still Stuck in the Lab - Bioengineer.org

1: Experts Call for Stronger Healthcare AI Oversight - Specialty Pharmacy Continuum

2: Americans want transparency when AI is used in their healthcare - Pew Research Center

3: Using AI in healthcare - wsaw.com

4: Wisconsin lawmakers weigh AI regulation in healthcare - WMTV 15 NEWS

5: AI Tracker: Forus lands $150M, reports $3B valuation - modernhealthcare.com

6: The hidden costs of ungoverned AI: What leading health systems are doing instead - Wolters Kluwer

7: Ferris State’s Tech Week Grand Rapids sessions focus on the growing role of artificial intelligence - ferris.edu

8: AI’s role in healthcare expanding as tech develops - WisBusiness

9: AI | Will AI healthcare be China’s new global growth engine? - South China Morning Post

10: Medicine Needs to Get Serious About AI - The Atlantic

11: In real-world test, an AI model did better than doctors at diagnosing patients - NPR

In [6]:
# Themes were manually assigned after reading each headline

themes = [
    "Translation",
    "Oversight",
    "Transparency",
    "Adoption",
    "Regulation",
    "Investment",
    "Governance",
    "Education",
    "Expansion",
    "Growth",
    "Caution",
    "Performance",
    "Diagnosis",
    "Autonomy",
    "Reasoning",
    "Applications",
    "Limitations",
    "Diagnosis",
    "Rare Disease",
    "Agents"
]

df["Theme"] = themes

df[
    [
        "Text",
        "AFINN Score",
        "AFINN Sentiment",
        "DistilBERT Sentiment",
        "DistilBERT Confidence",
        "Theme"
    ]
]

,Text,AFINN Score,AFINN Sentiment,DistilBERT Sentiment,DistilBERT Confidence,Theme
0,"AI in Healthcare Poised to Transform Medicine, But Most Tools Still Stuck in the Lab - Bioengineer.org",-4.0,Negative,NEGATIVE,0.9977,Translation
1,Experts Call for Stronger Healthcare AI Oversight - Specialty Pharmacy Continuum,1.0,Positive,POSITIVE,0.5517,Oversight
2,Americans want transparency when AI is used in their healthcare - Pew Research Center,1.0,Positive,NEGATIVE,0.9589,Transparency
3,Using AI in healthcare - wsaw.com,0.0,Neutral,NEGATIVE,0.9817,Adoption
4,Wisconsin lawmakers weigh AI regulation in healthcare - WMTV 15 NEWS,0.0,Neutral,NEGATIVE,0.7790,Regulation
5,"AI Tracker: Forus lands $150M, reports $3B valuation - modernhealthcare.com",0.0,Neutral,NEGATIVE,0.9967,Investment
6,The hidden costs of ungoverned AI: What leading health systems are doing instead - Wolters Kluwer,2.0,Positive,NEGATIVE,0.9987,Governance
7,Ferris State’s Tech Week Grand Rapids sessions focus on the growing role of artificial intelligence - ferris.edu,4.0,Positive,POSITIVE,0.9912,Education
8,AI’s role in healthcare expanding as tech develops - WisBusiness,0.0,Neutral,POSITIVE,0.7549,Expansion
9,AI | Will AI healthcare be China’s new global growth engine? - South China Morning Post,2.0,Positive,NEGATIVE,0.9566,Growth


In [8]:
# Create the final Excel file

final_df = df[
    [
        "Keyword",
        "Text",
        "AFINN Score",
        "AFINN Sentiment",
        "DistilBERT Sentiment",
        "DistilBERT Confidence",
        "Theme"
    ]
]

final_df.to_excel(
    "Homework2_Sentiment_Analysis.xlsx",
    index=False
)

print("Excel file created successfully!")

Excel file created successfully!


In [9]:
print("AFINN results:")
print(df["AFINN Sentiment"].value_counts())

print("\nDistilBERT results:")
print(df["DistilBERT Sentiment"].value_counts())

# Compare the two methods
df["Methods Agree"] = (
    df["AFINN Sentiment"].str.upper() ==
    df["DistilBERT Sentiment"]
)

print("\nNumber of exact agreements:")
print(df["Methods Agree"].sum())

print("\nNumber of disagreements:")
print((~df["Methods Agree"]).sum())

AFINN results:
AFINN Sentiment
Neutral     10
Positive     9
Negative     1
Name: count, dtype: int64

DistilBERT results:
DistilBERT Sentiment
NEGATIVE    12
POSITIVE     8
Name: count, dtype: int64

Number of exact agreements:
5

Number of disagreements:
15
